<a href="https://colab.research.google.com/github/lapistach/FeCapSNN/blob/main/code/Inference_CIFARDVS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [129]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [130]:
%%bash
git clone https://github.com/fangwei123456/spikingjelly.git /content/spikingjelly
cd /content/spikingjelly
git reset --hard 73f94ab983d0167623015537f7d4460b064cfca1
pip install .
pip install tensorboard -q

HEAD is now at 73f94ab9 增加detach reset的选项
Processing /content/spikingjelly
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for spikingjelly: filename=spikingjelly-0.0.0.0.1-py3-none-any.whl size=120107 sha256=b3a0a087d5997823df8ceff2a1bec6974a9025fc678d150fb06a833163a422a5
  Stored in directory: /tmp/pip-ephem-wheel-cache-78824a7n/wheels/8b/7c/eb/eaababa3ddd15578fcabcc0376ba9473acfed277adc5a871cd
Successfully built spikingjelly
  Attempting uninstall: spikingjelly
    Found existing installation: spikingjelly 0.0.0.0.1
    Uninstalling spikingjelly-0.0.0.0.1:
      Successfully uninstalled spikingjelly-0.0.0.0.1


fatal: destination path '/content/spikingjelly' already exists and is not an empty directory.


In [131]:
init_path = '/usr/local/lib/python3.12/dist-packages/spikingjelly/datasets/__init__.py'

new_content = """from .cifar10_dvs import CIFAR10DVS
"""

with open(init_path, 'w') as f:
    f.write(new_content)

print("Done — __init__.py now only imports CIFAR10DVS")

Done — __init__.py now only imports CIFAR10DVS


In [132]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from spikingjelly.clock_driven import functional, layer, surrogate, accelerating
from spikingjelly.clock_driven.neuron import BaseNode, LIFNode
from torchvision import transforms
import math

class PLIFNode(BaseNode):
    def __init__(self, init_tau=2.0, v_threshold=1.0, v_reset=0.0, detach_reset=True, surrogate_function=surrogate.ATan(), monitor_state=False):
        super().__init__(v_threshold, v_reset, surrogate_function, detach_reset, monitor_state)
        init_w = - math.log(init_tau - 1)
        self.w = nn.Parameter(torch.tensor(init_w, dtype=torch.float))

    def forward(self, dv: torch.Tensor):
        if self.v_reset is None:
            self.v += (dv - self.v) * self.w.sigmoid()
        else:
            self.v += (dv - (self.v - self.v_reset)) * self.w.sigmoid()
        return self.spiking()

    def tau(self):
        return 1 / self.w.data.sigmoid().item()

    def extra_repr(self):
        return f'v_threshold={self.v_threshold}, v_reset={self.v_reset}, tau={self.tau()}'

def create_conv_sequential(in_channels, out_channels, number_layer, init_tau, use_plif, use_max_pool, alpha_learnable, detach_reset):
    # 首层是in_channels-out_channels
    # 剩余number_layer - 1层都是out_channels-out_channels
    conv = [
        nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
        nn.MaxPool2d(2, 2) if use_max_pool else nn.AvgPool2d(2, 2)
    ]

    for i in range(number_layer - 1):
        conv.extend([
            nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
            nn.MaxPool2d(2, 2) if use_max_pool else nn.AvgPool2d(2, 2)
        ])
    return nn.Sequential(*conv)


def create_2fc(channels, h, w, dpp, class_num, init_tau, use_plif, alpha_learnable, detach_reset):
    return nn.Sequential(
        nn.Flatten(),
        layer.Dropout(dpp),
        nn.Linear(channels * h * w, channels * h * w // 4, bias=False),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
        layer.Dropout(dpp, dropout_spikes=True),
        nn.Linear(channels * h * w // 4, class_num * 10, bias=False),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
    )


class StaticNetBase(nn.Module):
    def __init__(self, T, init_tau, use_plif, use_max_pool, alpha_learnable, detach_reset):
        super().__init__()
        self.T = T
        self.init_tau = init_tau
        self.use_plif = use_plif
        self.use_max_pool = use_max_pool
        self.alpha_learnable = alpha_learnable
        self.detach_reset = detach_reset
        self.train_times = 0
        self.max_test_accuracy = 0
        self.epoch = 0
        self.static_conv = None
        self.conv = None
        self.fc = None
        self.boost = nn.AvgPool1d(10, 10)

    def forward(self, x):
        x = self.static_conv(x)
        out_spikes_counter = self.boost(self.fc(self.conv(x)).unsqueeze(1)).squeeze(1)
        for t in range(1, self.T):
            out_spikes_counter += self.boost(self.fc(self.conv(x)).unsqueeze(1)).squeeze(1)

        return out_spikes_counter


class NeuromorphicNet(nn.Module):
    def __init__(self, T, init_tau, use_plif, use_max_pool, alpha_learnable, detach_reset):
        super().__init__()
        self.T = T
        self.init_tau = init_tau
        self.use_plif = use_plif
        self.use_max_pool = use_max_pool
        self.alpha_learnable = alpha_learnable
        self.detach_reset = detach_reset

        self.train_times = 0
        self.max_test_accuracy = 0
        self.epoch = 0
        self.conv = None
        self.fc = None
        self.boost = nn.AvgPool1d(10, 10)

    def forward(self, x):
        x = x.permute(1, 0, 2, 3, 4)  # [T, N, 2, *, *]
        out_spikes_counter = self.boost(self.fc(self.conv(x[0])).unsqueeze(1)).squeeze(1)
        for t in range(1, x.shape[0]):
            out_spikes_counter += self.boost(self.fc(self.conv(x[t])).unsqueeze(1)).squeeze(1)
        return out_spikes_counter


class CIFAR10DVSNet(NeuromorphicNet):
    def __init__(self, T, init_tau, use_plif, use_max_pool, alpha_learnable, channels, number_layer, detach_reset):
        super().__init__(T=T, init_tau=init_tau, use_plif=use_plif, use_max_pool=use_max_pool, alpha_learnable=alpha_learnable, detach_reset=detach_reset)
        w = 128
        h = 128
        self.conv = create_conv_sequential(2, channels, number_layer=number_layer, init_tau=init_tau, use_plif=use_plif,
                                           use_max_pool=use_max_pool, alpha_learnable=alpha_learnable, detach_reset=detach_reset)
        self.fc = create_2fc(channels=channels, w=w >> number_layer, h=h >> number_layer, dpp=0.5, class_num=10,
                             init_tau=init_tau, use_plif=use_plif, alpha_learnable=alpha_learnable, detach_reset=detach_reset)


class Interpolate(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.args = args
        self.kwargs = kwargs
    def forward(self, x):
        return F.interpolate(x, *self.args, **self.kwargs)

class ASLDVSNet(NeuromorphicNet):
    def __init__(self, T, init_tau, use_plif, use_max_pool, alpha_learnable, detach_reset, channels, number_layer):
        super().__init__(T=T, init_tau=init_tau, use_plif=use_plif, use_max_pool=use_max_pool, alpha_learnable=alpha_learnable, detach_reset=detach_reset)
        # input size 256 * 256
        w = 256
        h = 256

        self.conv = nn.Sequential(
            Interpolate(size=256, mode='bilinear'),
        )

In [133]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from spikingjelly.clock_driven import functional
import torchvision
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import os
import argparse
import time
import sys

torch.backends.cudnn.benchmark = True
_seed_ = 2020
torch.manual_seed(_seed_)  # use torch.manual_seed() to seed the RNG for all devices (both CPU and CUDA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(_seed_)

In [134]:
init_tau = 2.0
batch_size = 16
learning_rate = 1e-3
T_max = 64
use_plif = True
alpha_learnable = False
use_max_pool = True
device = 'cuda:0'
dataset_dir = '/content/drive/MyDrive/datasets/CIFAR10DVS'
log_dir_prefix = '/content/drive/MyDrive/logs'
T = 20
max_epoch = 256
detach_reset = True
number_layer = 5
channels = 128
split_by = 'number'
normalization = None
class_num = 10

In [135]:
train_data_loader = torch.load('/content/drive/MyDrive/datasets/CIFAR10DVS/train.pt', weights_only=False)
test_data_loader = torch.load('/content/drive/MyDrive/datasets/CIFAR10DVS/test.pt', weights_only=False)

In this next cell, load the checkpoint index that is found in post-processing.

In [136]:
dir_name = f'{'CIFAR10DVS'}_init_tau_{init_tau}_use_plif_{use_plif}_use_max_pool_{use_max_pool}_T_{T}_c_{channels}_n_{number_layer}_split_by_{split_by}_normalization_{normalization}_detach_reset_{detach_reset}'
log_dir = os.path.join(log_dir_prefix, dir_name)
pt_dir = os.path.join(log_dir_prefix, 'pt_' + dir_name)
check_point_path = os.path.join(pt_dir, 'check_point_105.pt')
check_point = torch.load(check_point_path, map_location=device, weights_only=False)
net = check_point['net']

In [137]:
optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max)

In [138]:
optimizer.load_state_dict(check_point['optimizer'])
scheduler.load_state_dict(check_point['scheduler'])

In [139]:
tau_list = []
for m in net.modules():
  if isinstance(m, PLIFNode):
    beta = torch.sigmoid(m.w).item()
    tau = 1/beta
    tau_list.append(tau)
print(tau_list)

[1.638416320162564, 1.7332377242800059, 1.4571366161983528, 1.2703955606618753, 1.5259945555339989, 3.15613397096063, 1.3087698307046072]


This next cell is composed of helper functions to find the corresponding taus to fecaps.

In [140]:
import numpy as np
from sklearn.cluster import KMeans

# Compute on/off ratio, relative ratios inside the list, and then how to map from one to the other
def on_off_ratio(l: list):
  return max(l)/min(l)

def relative_ratios(l: list):
  ratio_list = []
  for i in range(len(l)):
    ratio_list.append((l[i]-min(l))/(max(l)-min(l)))
  return ratio_list

def corresponding_tau_to_fecap(analog_elements_list: list, tmin: float):
  new_list = []
  relative_list = relative_ratios(analog_elements_list)
  ratio_on_off = on_off_ratio(analog_elements_list)
  for i in range(len(relative_list)):
    value = tmin*(1+relative_list[i]*(ratio_on_off-1))
    new_list.append(value)
  return new_list

This next cell converts the taus into analog values that respect the ratios within our fecap list.

In [141]:
from numpy._core.fromnumeric import argmax
# First we inventory our fecaps
fecap_list = [47.4, 48, 48.8, 49.7, 50.7, 51.4]
#fecap_list = [47,48.3,49.6,51]
#fecap_list = [47,49,51]
#fecap_list = [47,51]

tau_min = sum(tau_list)/len(tau_list) ## "when in doubt, take the mean"
tau_min = min(tau_list) ## "when in doubt, take the min"
tau_min = 1.4 ## manual choose from printed tau_list

# FailSafe
tau_min_inf = min(tau_list)
tau_min_sup = max(tau_list)/on_off_ratio(fecap_list)
if tau_min > tau_min_sup:
  raise Exception('tau_min too big, stopping execution.')
elif tau_min < tau_min_inf:
  raise Exception('tau_min too small, stopping execution.')

# Find the corresponding tau values for each group of tau
corresponding_list = corresponding_tau_to_fecap(fecap_list, tau_min)

# Comment out if you don't want to keep the leaky integrate and fire behavior of the highest tau neuron
# replace the max of tau_list by the second max
#remember_argmax=argmax(tau_list)
#remember_max = max(tau_list)
#new_list = tau_list.copy()
#new_list.remove(max(tau_list))
#second_max = max(new_list)
#tau_list[argmax(tau_list)]= second_max


# Cluster the tau values into the number of available analog values

X = np.array(tau_list).reshape(-1, 1)
kmeans = KMeans(n_clusters=len(fecap_list)).fit(X)
order = np.argsort(kmeans.cluster_centers_.flatten())
label_mapping = np.zeros_like(order)
label_mapping[order] = np.arange(len(order))
ordered_labels = label_mapping[kmeans.labels_]

# Write the new tau list
new_tau_list = [corresponding_list[i] for i in ordered_labels]
print(new_tau_list)
# Comment out if you don't want to keep the leaky integrate and fire behavior of the highest tau neuron
#new_tau_list[remember_argmax] = remember_max

[1.467932489451477, 1.49746835443038, 1.4177215189873418, 1.4, 1.4413502109704641, 1.5181434599156118, 1.4]


Replacing and printing for sanity check.

In [142]:
i = 0
for m in net.modules():
  if isinstance(m, PLIFNode):
      print('before', m.w.item())
      new_w = -math.log(new_tau_list[i] - 1)
      i+=1
      m.w.data = torch.tensor(new_w, dtype=m.w.dtype, device=m.w.device)
      print('after', m.w.item())

before 0.44876474142074585
after 0.7594312429428101
before 0.3102853000164032
after 0.698223352432251
before 0.7827728986740112
after 0.8729403018951416
before 1.307869791984558
after 0.9162907600402832
before 0.6424645781517029
after 0.8179165720939636
before -0.7683167457580566
after 0.6575031280517578
before 1.1751593351364136
after 0.9162907600402832


Perform inference.

In [143]:
net.eval()
with torch.no_grad():
    test_sum = 0
    correct_sum = 0
    for img, label in test_data_loader:
        img = img.to(device)
        label = label.to(device)
        out_spikes_counter = net(img)
        correct_sum += (out_spikes_counter.argmax(dim=1) == label).float().sum().item()
        test_sum += label.numel()
        functional.reset_net(net)
    test_accuracy = correct_sum / test_sum
    print('test_accuracy', test_accuracy)



test_accuracy 0.73
